# AI-Enhanced E-Commerce Platform
## Machine Learning Module: K-Nearest Neighbors (KNN) Product Recommendation System

### Academic Project Demonstration
This notebook demonstrates the end-to-end development of the recommendation system:
1. **Dataset Loading & Preprocessing**
2. **Feature Engineering & Matrix Construction**
3. **K-Nearest Neighbors (KNN) Collaborative & Content Model Training**
4. **Rigorous Model Evaluation (Precision@K, Recall@K, Coverage)**
5. **Personalized and Cold-Start Recommendation Generation**
6. **Model Serialization with Joblib for FastAPI Backend Deployment**

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
import joblib

# Set dataset path
DATASET_DIR = Path('../dataset')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('All libraries loaded successfully!')

## 1. Dataset Loading & Inspection
We load two primary datasets:
- `products.csv`: Contains product catalogs, categories, prices, ratings, and stock.
- `user_interactions.csv`: Contains user browsing, search, cart, and purchase activities.

In [2]:
products_df = pd.read_csv(DATASET_DIR / 'products.csv')
interactions_df = pd.read_csv(DATASET_DIR / 'user_interactions.csv')

print(f'Total Products: {len(products_df)}')
print(f'Total User Interactions: {len(interactions_df)}')
print(f'Unique Users: {interactions_df["user_id"].nunique()}')

products_df.head(5)

In [3]:
interactions_df.head(5)

## 2. Feature Engineering & Matrix Construction
We build two complementary representations:
1. **User-Item Matrix**: Captures historical preference signals (ratings, views, carts, purchases).
2. **Item-Feature Matrix**: Combines categorical one-hot encoding (category), price scaling, rating normalization, and log-transformed review popularity.

In [4]:
# 1. User-Item Interaction Matrix
user_item_matrix = interactions_df.pivot_table(
    index='user_id',
    columns='product_id',
    values='rating',
    aggfunc='max',
    fill_value=0.0
)

# Ensure all product columns are present
for pid in products_df['product_id']:
    if pid not in user_item_matrix.columns:
        user_item_matrix[pid] = 0.0
user_item_matrix = user_item_matrix[sorted(user_item_matrix.columns)]

print('User-Item Matrix Shape:', user_item_matrix.shape)
user_item_matrix.iloc[:5, :5]

In [5]:
# 2. Item Feature Matrix
cat_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_encoded = cat_encoder.fit_transform(products_df[['category_name']])
cat_cols = [f'cat_{c}' for c in cat_encoder.categories_[0]]
cat_df = pd.DataFrame(cat_encoded, columns=cat_cols, index=products_df.index)

# Scale numerical features (price, rating, log popularity)
scaler = MinMaxScaler()
num_features = products_df[['price', 'rating']].copy()
num_features['log_reviews'] = np.log1p(products_df['num_reviews'])
scaled_nums = scaler.fit_transform(num_features)
num_df = pd.DataFrame(scaled_nums, columns=['price_scaled', 'rating_scaled', 'pop_scaled'], index=products_df.index)

item_feature_matrix = pd.concat([products_df[['product_id']], cat_df, num_df], axis=1).set_index('product_id')
print('Item-Feature Matrix Shape:', item_feature_matrix.shape)
item_feature_matrix.head(5)

## 3. Training the K-Nearest Neighbors (KNN) Models
We use Scikit-Learn's `NearestNeighbors` with `metric='cosine'` and `algorithm='brute'`:
$$\text{Cosine Distance} = 1 - \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|}$$

- **Item-KNN**: Computes item similarity vectors based on features and interactions.
- **User-KNN**: Computes user similarity vectors for collaborative filtering.

In [6]:
# Fit Item-KNN
k_items = min(6, len(item_feature_matrix))
item_knn = NearestNeighbors(n_neighbors=k_items, metric='cosine', algorithm='brute')
item_knn.fit(item_feature_matrix.values)

# Fit User-KNN
k_users = min(5, len(user_item_matrix))
user_knn = NearestNeighbors(n_neighbors=k_users, metric='cosine', algorithm='brute')
user_knn.fit(user_item_matrix.values)

print(f'Item-KNN successfully trained with k={k_items}')
print(f'User-KNN successfully trained with k={k_users}')

## 4. Model Evaluation
To prevent data leakage, we perform a user-stratified Train/Test split on interaction histories (80% train, 20% test).
We compute:
- **Precision@K**: Proportion of recommended items that were relevant.
- **Recall@K**: Proportion of relevant items that were successfully recommended.
- **Hit Rate@K**: Percentage of users who received at least one relevant recommendation.
- **Catalog Coverage**: Percentage of total items recommended across all users.

In [7]:
np.random.seed(42)
all_pids = set(products_df['product_id'])
train_rows, test_rows = [], []

for uid, group in interactions_df.groupby('user_id'):
    if len(group) < 3:
        train_rows.append(group)
        continue
    shuffled = group.sample(frac=1.0, random_state=42)
    n_test = max(1, int(len(shuffled) * 0.25))
    test_rows.append(shuffled.iloc[:n_test])
    train_rows.append(shuffled.iloc[n_test:])

eval_train_df = pd.concat(train_rows)
eval_test_df = pd.concat(test_rows)

eval_train_matrix = eval_train_df.pivot_table(index='user_id', columns='product_id', values='rating', aggfunc='max', fill_value=0.0)
for pid in all_pids:
    if pid not in eval_train_matrix.columns:
        eval_train_matrix[pid] = 0.0
eval_train_matrix = eval_train_matrix[sorted(eval_train_matrix.columns)]

eval_knn = NearestNeighbors(n_neighbors=5, metric='cosine', algorithm='brute')
eval_knn.fit(eval_train_matrix.values)

ground_truth = eval_test_df.groupby('user_id')['product_id'].apply(set).to_dict()

for k in [3, 5, 10]:
    precisions, recalls, hits, rec_all = [], [], [], set()
    for uid, true_items in ground_truth.items():
        if uid not in eval_train_matrix.index:
            continue
        u_vec = eval_train_matrix.loc[uid].values.reshape(1, -1)
        dists, idxs = eval_knn.kneighbors(u_vec, n_neighbors=5)
        already = set(eval_train_df[eval_train_df['user_id'] == uid]['product_id'])
        scores = {}
        for d, n_idx in zip(dists[0][1:], idxs[0][1:]):
            sim = 1.0 - d
            n_uid = eval_train_matrix.index[n_idx]
            for pid, r in eval_train_matrix.loc[n_uid].items():
                if pid not in already and r > 0:
                    scores[pid] = scores.get(pid, 0.0) + (sim * r)
        recs = [pid for pid, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]]
        if len(recs) < k:
            recs.extend([p for p in all_pids if p not in already and p not in recs][:k - len(recs)])
        rec_all.update(recs)
        inter = set(recs).intersection(true_items)
        precisions.append(len(inter) / float(k))
        recalls.append(len(inter) / float(len(true_items)) if true_items else 0.0)
        hits.append(1.0 if inter else 0.0)
    cov = (len(rec_all) / len(all_pids)) * 100.0
    print(f'Results for K={k}: Precision@{k}={np.mean(precisions):.4f} | Recall@{k}={np.mean(recalls):.4f} | HitRate@{k}={np.mean(hits):.4f} | Coverage={cov:.1f}%')

## 5. Live Recommendation Demonstrations
### A. Similar Products via Item-KNN

In [8]:
query_pid = 1  # AuraSound Pro Wireless Headphones
q_row = products_df[products_df['product_id'] == query_pid].iloc[0]
print(f'Target Product: {q_row["name"]} ({q_row["category_name"]})')

pid_index = list(item_feature_matrix.index).index(query_pid)
distances, indices = item_knn.kneighbors([item_feature_matrix.iloc[pid_index].values], n_neighbors=4)

print('\nKNN Similar Products:')
for d, n_idx in zip(distances[0][1:], indices[0][1:]):
    sim_pid = list(item_feature_matrix.index)[n_idx]
    row = products_df[products_df['product_id'] == sim_pid].iloc[0]
    sim_pct = (1.0 - d) * 100
    print(f' -> #{sim_pid}: {row["name"]} | Category: {row["category_name"]} | Similarity: {sim_pct:.1f}%')

### B. Personalized Recommendations via Collaborative User-KNN

In [9]:
test_user = 1
print(f'User #{test_user} Past Interactions:')
u_history = interactions_df[interactions_df['user_id'] == test_user]
for _, r in u_history.iterrows():
    pname = products_df[products_df['product_id'] == r['product_id']]['name'].values[0]
    print(f' -> {pname} (Action: {r["interaction_type"]}, Score: {r["rating"]})')

# Recommend for User 1
u_vector = user_item_matrix.loc[test_user].values.reshape(1, -1)
dists, idxs = user_knn.kneighbors(u_vector, n_neighbors=4)
already_interacted = set(u_history['product_id'])

scores = {}
for d, n_idx in zip(dists[0][1:], idxs[0][1:]):
    sim = 1.0 - d
    n_uid = user_item_matrix.index[n_idx]
    for pid, r in user_item_matrix.loc[n_uid].items():
        if pid not in already_interacted and r > 0:
            scores[pid] = scores.get(pid, 0.0) + (sim * r)

print('\nPersonalized Recommendations for User #1:')
for pid, s in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:3]:
    prow = products_df[products_df['product_id'] == pid].iloc[0]
    print(f' -> Recommended: {prow["name"]} | Category: {prow["category_name"]} | Score: {s:.2f}')

## 6. Model Serialization with Joblib
We serialize the complete artifact bundle so the FastAPI backend can load and serve recommendations with sub-millisecond latency.

In [10]:
bundle = {
    'item_knn': item_knn,
    'user_knn': user_knn,
    'user_item_matrix': user_item_matrix,
    'item_feature_matrix': item_feature_matrix,
    'product_df': products_df,
    'category_encoder': cat_encoder,
    'scaler': scaler,
    'product_ids': list(item_feature_matrix.index)
}

model_save_path = MODELS_DIR / 'knn_recommender.joblib'
joblib.dump(bundle, model_save_path)
print(f'Successfully serialized trained KNN model to: {model_save_path.resolve()}')